## Modeling — Corey Williams
Picks up after Eli's MySQL loading cells. Requires `df` (train) and `test_df` (test) to already be defined.

In [ ]:
# Setup — imports and data preparation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_preprocessing import preprocess_dataframe

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, roc_auc_score
)

# Extract target variable before preprocessing
y_train = df['Attrition'].map({'Left': 1, 'Stayed': 0})
y_test  = test_df['Attrition'].map({'Left': 1, 'Stayed': 0})

# Preprocess features — scales numeric columns and one-hot encodes categorical ones
drop_cols = ['Attrition', 'Employee ID']
X_train = preprocess_dataframe(df.drop(columns=drop_cols), scale_numeric=True)
X_test  = preprocess_dataframe(test_df.drop(columns=drop_cols), scale_numeric=True)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"Training features : {X_train.shape}")
print(f"Test features     : {X_test.shape}")
print(f"Attrition rate (train): {y_train.mean():.2%}")
print(f"Attrition rate (test) : {y_test.mean():.2%}")

In [ ]:
# Logistic Regression
# Estimates the probability an employee leaves using a weighted sum of features.
# C=1 is the regularization strength — controls how much the model penalizes large weights.
lr = LogisticRegression(C=1, solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# Cross-validation — trains and tests on 5 different splits to get a reliable score
cv_auc_lr = cross_val_score(lr, X_train, y_train, cv=5, scoring='roc_auc')
print(f"CV ROC-AUC: {cv_auc_lr.mean():.4f} ± {cv_auc_lr.std():.4f}")

# Evaluate on the held-out test set
lr_preds = lr.predict(X_test)
lr_probs = lr.predict_proba(X_test)[:, 1]
print(f"Test Accuracy : {accuracy_score(y_test, lr_preds):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, lr_probs):.4f}")
print(classification_report(y_test, lr_preds, target_names=['Stayed', 'Left']))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, lr_preds, display_labels=['Stayed', 'Left'], cmap='Blues'
)
plt.title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Random Forest
# Builds many decision trees and takes a majority vote.
# Captures non-linear patterns that logistic regression may miss.
rf = RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

cv_auc_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='roc_auc')
print(f"CV ROC-AUC: {cv_auc_rf.mean():.4f} ± {cv_auc_rf.std():.4f}")

rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]
print(f"Test Accuracy : {accuracy_score(y_test, rf_preds):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, rf_probs):.4f}")
print(classification_report(y_test, rf_preds, target_names=['Stayed', 'Left']))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, rf_preds, display_labels=['Stayed', 'Left'], cmap='Oranges'
)
plt.title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

# Feature importances — which features most influenced predictions
feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot(kind='barh', ax=ax, color='#ff7f0e', edgecolor='black')
ax.invert_yaxis()
ax.set_title('Top 15 Feature Importances — Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# Results comparison
results = pd.DataFrame({
    'Model'         : ['Logistic Regression', 'Random Forest'],
    'CV ROC-AUC'    : [f"{cv_auc_lr.mean():.4f} ± {cv_auc_lr.std():.4f}",
                       f"{cv_auc_rf.mean():.4f} ± {cv_auc_rf.std():.4f}"],
    'Test Accuracy' : [f"{accuracy_score(y_test, lr_preds):.4f}",
                       f"{accuracy_score(y_test, rf_preds):.4f}"],
    'Test ROC-AUC'  : [f"{roc_auc_score(y_test, lr_probs):.4f}",
                       f"{roc_auc_score(y_test, rf_probs):.4f}"],
})
print(results.to_string(index=False))

winner = 'Random Forest' if roc_auc_score(y_test, rf_probs) > roc_auc_score(y_test, lr_probs) \
         else 'Logistic Regression'
print(f"\nBest model by ROC-AUC: {winner}")